# Translate Comparison Mabsa

Research code accompanying [Şeker (2026), Expert Systems](https://doi.org/10.1111/exsy.70300).
See the [repository README](../README.md) for dependencies, dataset preparation and citation.

This is an exploratory notebook: select the intended dataset and experiment cells before
running. Local intermediate datasets and manual notebook state are required. Stored
outputs and execution counts have been removed. Model and API identifiers retain the
historical experiment settings; check availability before calling external services.

Both historical translation notebooks currently load `data/MARC_translate_sentiment_topic_all`; the M-ABSA filename does not establish a separate M-ABSA experiment.


In [ ]:
from pathlib import Path
import os

# Keep relative data/model paths anchored at the repository root.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)


In [ ]:
%pip install sacremoses

In [ ]:
import pandas as pd
import requests
from tqdm import tqdm

In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("data/MARC_translate_sentiment_topic_all")
df = dataset.to_pandas()

In [ ]:
sampled_df = df[df['language'] != 'en'].groupby('language').apply(lambda x: x.sample(n=100, random_state=42)).reset_index(drop=True)

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
from tqdm import tqdm

LANG_MODEL_MAP = {
    "de": "Helsinki-NLP/opus-mt-de-en",
    "fr": "Helsinki-NLP/opus-mt-fr-en",
    "es": "Helsinki-NLP/opus-mt-es-en",
    "zh": "Helsinki-NLP/opus-mt-zh-en",
    "ja": "Helsinki-NLP/opus-mt-ja-en",
}

loaded_models = {}

def helsinki_translate_multilang(df):
    results = []

    for lang, group in tqdm(df.groupby("language"), desc="Helsinki Translation", total=df["language"].nunique()):
        model_name = LANG_MODEL_MAP.get(lang)
        if not model_name:
            results.extend([None] * len(group))
            continue

        if model_name not in loaded_models:
            tokenizer = MarianTokenizer.from_pretrained(model_name)
            model = MarianMTModel.from_pretrained(model_name)
            loaded_models[model_name] = (tokenizer, model)
        else:
            tokenizer, model = loaded_models[model_name]

        texts = group["text"].tolist()
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
        translated = model.generate(**inputs)
        outputs = tokenizer.batch_decode(translated, skip_special_tokens=True)
        results.extend(outputs)

    return results


In [ ]:
sampled_df["helsinki_translation"] = helsinki_translate_multilang(sampled_df)

In [ ]:
# Helsinki örneği
model.save_pretrained("models/helsinki_opus_mt_de_en")
tokenizer.save_pretrained("models/helsinki_opus_mt_de_en")    

# Helsinki örneği
model.save_pretrained("models/helsinki_opus_mt_fr_en")
tokenizer.save_pretrained("models/helsinki_opus_mt_fr_en")    

# Helsinki örneği
model.save_pretrained("models/helsinki_opus_mt_es_en")
tokenizer.save_pretrained("models/helsinki_opus_mt_es_en")    

# Helsinki örneği
model.save_pretrained("models/helsinki_opus_mt_zh_en")
tokenizer.save_pretrained("models/helsinki_opus_mt_zh_en")    

# Helsinki örneği
model.save_pretrained("models/helsinki_opus_mt_ja_en")
tokenizer.save_pretrained("models/helsinki_opus_mt_ja_en")    

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Set the Deepl API key
DEEPL_API_KEY = os.getenv("DEEPL_API_KEY")

In [ ]:
import requests

def deepl_translate(text, source_lang="auto", target_lang="en"):
    url = "https://api-free.deepl.com/v2/translate"
    headers = {
        "Authorization": f"DeepL-Auth-Key {DEEPL_API_KEY}"
    }
    params = {
        "text": text,
        "source_lang": source_lang.upper() if source_lang != "auto" else None,
        "target_lang": target_lang.upper()
    }

    try:
        response = requests.post(url, headers=headers, data=params)
        if response.status_code == 200:
            return response.json()["translations"][0]["text"]
        else:
            return f"ERROR {response.status_code}: {response.text}"
    except Exception as e:
        return f"ERROR: {str(e)}"


In [ ]:
from tqdm import tqdm
import time

def run_deepl_batched_translation(df: pd.DataFrame, text_col: str = "text", lang_col: str = "language") -> pd.DataFrame:
    deepl_translations = []
    deepl_errors = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="DeepL Translation"):
        text = row[text_col]
        source_lang = row[lang_col]

        result = deepl_translate(text, source_lang=source_lang, target_lang="en")

        if result.startswith("ERROR"):
            deepl_translations.append(None)
            deepl_errors.append(result)
        else:
            deepl_translations.append(result)
            deepl_errors.append(None)

        time.sleep(1.1)  # 🔄 60 istek/dakika sınırı (free plan için güvenli)

    df["translated_deepl"] = deepl_translations
    df["deepl_error"] = deepl_errors

    return df


In [ ]:
translated_df = run_deepl_batched_translation(sampled_df)

In [ ]:
%pip install transformers sentencepiece


In [ ]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

# Model ve tokenizer yükle
model_name = "facebook/m2m100_418M"
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

def m2m_translate(text: str, source_lang: str, target_lang: str = "en") -> str:
    try:
        tokenizer.src_lang = source_lang
        encoded = tokenizer(text, return_tensors="pt")
        generated_tokens = model.generate(
            **encoded,
            forced_bos_token_id=tokenizer.get_lang_id(target_lang),
            max_length=512
        )
        return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    except Exception as e:
        return f"❌ ERROR: {e}"


In [ ]:
example = "J'aime ce produit, mais la livraison était lente."
translated = m2m_translate(example, source_lang="fr", target_lang="en")
print(translated)

In [ ]:
# M2M100 örneği
model.save_pretrained("models/m2m100_418M")
tokenizer.save_pretrained("models/m2m100_418M")

In [ ]:
from tqdm import tqdm

def apply_m2m_translation(df, text_col="text", lang_col="language"):
    results = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Translating with M2M100"):
        text = row[text_col]
        src_lang = row[lang_col]
        if src_lang == "en":
            results.append(text)  # İngilizce ise olduğu gibi al
        else:
            translated = m2m_translate(text, source_lang=src_lang, target_lang="en")
            results.append(translated)
    return results

In [ ]:
sampled_df["translated_m2m"] = apply_m2m_translation(sampled_df)

In [ ]:
translated_df.to_csv("deepl_translation_results.csv", index=False)

In [ ]:
sampled_df.to_csv("deepl_translation_results_4model.csv", index=False)

In [ ]:
%pip install -U sentence-transformers

In [ ]:
sampled_df

In [ ]:
def tfidf_similarity(a, b):
    try:
        vect = TfidfVectorizer().fit([a, b])
        tfidf_matrix = vect.transform([a, b])
        return cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
    except:
        return None

In [ ]:
def closest_to_deepl(row):
    g_sim = tfidf_similarity(row['translated_text'], row['translated_deepl']) if pd.notnull(row['translated_text']) else -1
    h_sim = tfidf_similarity(row['helsinki_translation'], row['translated_deepl']) if pd.notnull(row['helsinki_translation']) else -1

    if g_sim > h_sim:
        return 'gemini'
    elif h_sim > g_sim:
        return 'helsinki'
    elif g_sim == h_sim and g_sim != -1:
        return 'equal'
    else:
        return 'unknown'

In [ ]:
sampled_df['closest_to_deepl'] = sampled_df.apply(closest_to_deepl, axis=1)

In [ ]:
print("🎯 Distribution vs Expert (DeepL):")
print(sampled_df['closest_to_deepl'].value_counts(normalize=True).round(3))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def compute_similarity_columns(df, reference_col='translated_deepl'):
    """
    Gemini, Helsinki, M2M çevirilerini DeepL referansına göre karşılaştırır.
    Sonuçları sim_gemini, sim_helsinki, sim_m2m olarak ekler.
    """
    models = {
        'sim_gemini': 'translated_text',
        'sim_helsinki': 'helsinki_translation',
        'sim_m2m': 'translated_m2m'
    }

    for sim_col, model_col in models.items():
        sims = []
        for a, b in zip(df[model_col], df[reference_col]):
            if pd.isna(a) or pd.isna(b):
                sims.append(-1)
            else:
                vect = TfidfVectorizer().fit([a, b])
                tfidf = vect.transform([a, b])
                sim_score = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]
                sims.append(sim_score)
        df[sim_col] = sims

    return df


In [ ]:
sampled_df = compute_similarity_columns(sampled_df)

In [ ]:
sampled_df['best_match'] = sampled_df.apply(
    lambda row: compare_similarity_three(
        row['sim_gemini'],
        row['sim_helsinki'],
        row['sim_m2m']
    ),
    axis=1
)

In [ ]:
print(sampled_df['best_match'].value_counts(dropna=False))


In [ ]:
print(sampled_df['best_match'].value_counts(normalize=True, dropna=False).round(3))


## tüm çevirileri tekrar skorla sıfırdan! işler karıştı. 